<a href="https://colab.research.google.com/github/Muqqadas30/fsdl-my-labs/blob/main/lab02/lab02b_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pytorch-lightning --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 25.3 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
import logging
import textwrap
from typing import Tuple

version = pl.__version__
print("PyTorch Lightning version:", version)

PyTorch Lightning version: 2.6.5


In [4]:
issubclass(pl.LightningModule, torch.nn.Module)

True

In [5]:
train_raw = datasets.FashionMNIST(root="./data", train=True, download=True)
valid_raw = datasets.FashionMNIST(root="./data", train=False, download=True)

x_train = train_raw.data.float().reshape(-1, 784) / 255.0
y_train = train_raw.targets
x_valid = valid_raw.data.float().reshape(-1, 784) / 255.0
y_valid = valid_raw.targets

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

100%|██████████| 26.4M/26.4M [00:00<00:00, 114MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 5.52MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 57.7MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 24.1MB/s]


In [6]:
class FashionClassifier(pl.LightningModule):

    def __init__(self):
        super().__init__()
        self.model = torch.nn.Linear(in_features=784, out_features=10)

    def forward(self, xs):
        return self.model(xs)

In [7]:
class FashionDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_ds = FashionDataset(x_train, y_train)
tdl = DataLoader(train_ds, batch_size=64, num_workers=0)

try:
    logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

    model = FashionClassifier()
    trainer = pl.Trainer(accelerator="auto", max_epochs=1)
    trainer.fit(model=model, train_dataloaders=tdl)

except Exception as error:
    print("Error:", *textwrap.wrap(str(error), 80), sep="\n\t")

finally:
    logging.getLogger("pytorch_lightning").setLevel(logging.INFO)

Error:
	No `training_step()` method defined. Lightning `Trainer` expects as minimum a
	`training_step()`, `train_dataloader()` and `configure_optimizers()` to be
	defined.


In [8]:
def training_step(self: pl.LightningModule, batch: Tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
    xs, ys = batch
    outs = self(xs)
    loss = F.cross_entropy(outs, ys)
    return loss

FashionClassifier.training_step = training_step

In [9]:
def configure_optimizers(self: FashionClassifier) -> torch.optim.Optimizer:
    optimizer = torch.optim.Adam(self.parameters(), lr=3e-4)
    return optimizer

FashionClassifier.configure_optimizers = configure_optimizers

In [10]:
model = FashionClassifier()

print("loss before training:", F.cross_entropy(model(x_train[:64]), y_train[:64]).item())

trainer = pl.Trainer(max_epochs=3, accelerator="auto")
trainer.fit(model=model, train_dataloaders=tdl)

print("loss after training:", F.cross_entropy(model(x_train[:64]), y_train[:64]).item())

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


loss before training: 2.3940281867980957


┏━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Linear │  7.9 K │ train │     0 │
└───┴───────┴────────┴────────┴───────┴───────┘

Trainable params: 7.9 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 7.9 K                                                                                                
Total estimated model params size (MB): 0.031                                                                      
Modules in train mode: 1                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=3` reached.


loss after training: 0.45484036207199097


In [11]:
class FashionDataModule(pl.LightningDataModule):

    def __init__(self, batch_size=32):
        super().__init__()
        self.batch_size = batch_size

    def setup(self, stage=None):
        if stage == "fit" or stage is None:
            self.train_dataset = FashionDataset(x_train, y_train)
            self.val_dataset = FashionDataset(x_valid, y_valid)

    def prepare_data(self):
        pass

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

In [12]:
model = FashionClassifier()
datamodule = FashionDataModule()

trainer = pl.Trainer(max_epochs=3, accelerator="auto")
trainer.fit(model=model, datamodule=datamodule)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.


┏━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Linear │  7.9 K │ train │     0 │
└───┴───────┴────────┴────────┴───────┴───────┘

Trainable params: 7.9 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 7.9 K                                                                                                
Total estimated model params size (MB): 0.031                                                                      
Modules in train mode: 1                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=3` reached.


In [13]:
class HelloWorldCallback(pl.Callback):

    def on_train_epoch_start(self, trainer, pl_module):
        print("👋 hello from the start of the training epoch!")

    def on_validation_epoch_end(self, trainer, pl_module):
        print("👋 hello from the end of the validation epoch!")

model = FashionClassifier()
datamodule = FashionDataModule()

trainer = pl.Trainer(max_epochs=2, accelerator="auto", callbacks=[HelloWorldCallback()])
trainer.fit(model=model, datamodule=datamodule)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Linear │  7.9 K │ train │     0 │
└───┴───────┴────────┴────────┴───────┴───────┘

Trainable params: 7.9 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 7.9 K                                                                                                
Total estimated model params size (MB): 0.031                                                                      
Modules in train mode: 1                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

👋 hello from the start of the training epoch!

👋 hello from the start of the training epoch!

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=2` reached.


In [14]:
def validation_step(self: pl.LightningModule, batch, batch_idx):
    xs, ys = batch
    outs = self(xs)
    loss = F.cross_entropy(outs, ys)
    preds = torch.argmax(outs, dim=1)
    acc = (preds == ys).float().mean()

    self.log("val_loss", loss, prog_bar=True)
    self.log("val_acc", acc, prog_bar=True)
    return loss

FashionClassifier.validation_step = validation_step

model = FashionClassifier()
datamodule = FashionDataModule()

trainer = pl.Trainer(max_epochs=3, accelerator="auto")
trainer.fit(model=model, datamodule=datamodule)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Linear │  7.9 K │ train │     0 │
└───┴───────┴────────┴────────┴───────┴───────┘

Trainable params: 7.9 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 7.9 K                                                                                                
Total estimated model params size (MB): 0.031                                                                      
Modules in train mode: 1                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=3` reached.
